In [ ]:
import os
import json
import pandas as pd
import traceback 

In [ ]:
from langchain_core.exceptions import ModelAPIError

print("ModelAPIError imported successfully!")

from langchain_openai import ChatOpenAI

print("ChatOpenAI imported successfully!")

In [ ]:
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env file

In [ ]:
KEY = os.getenv("OPENAI_KEY")
print("OPENAI_KEY loaded:", bool(KEY))

In [ ]:
llm=ChatOpenAI(openai_api_key=KEY, model="gpt-4o", temperature=0.5, max_tokens=2000, request_timeout=120)

In [ ]:
llm

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_community.callbacks.manager import get_openai_callback
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
import PyPDF2


In [ ]:
RESPONSE_JSON = {
    "1": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct": "correct answer",
    },
    "2": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct": "correct answer",
    },
    "3": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct": "correct answer",
    },
}

In [ ]:
TEMPLATE = """
Text: {text}
You are an expert MCQ maker. Given the above text, it is your job to \
create a quiz of {number} multiple choice questions for {subject} students in {tone} tone.
Make sure the questions are not repeated and check all the questions to be confirming the text as well.
Make sure to format your response like RESPONSE_JSON below and use it as a guide. \
Ensure make {number} MCQs
### RESPONSE_JSON
{response_json}

"""


In [ ]:
quiz_generation_prompt = PromptTemplate(
    input_variables=["text", "number", "subject", "tone", "response_json"],
    template=TEMPLATE
)

In [ ]:
quiz_chain = quiz_generation_prompt | llm | StrOutputParser()

In [ ]:
TEMPLATE2="""
You are an expert english grammarian and writer. Given a Multiple Choice Quiz for {subject} students.\
You need to evaluate the complexity of the question and give a complete analysis of the quiz. Only use at max 50 words for complexity analysis. 
if the quiz is not at per with the cognitive and analytical abilities of the students,\
update the quiz questions which needs to be changed and change the tone such that it perfectly fits the student abilities
Quiz_MCQs:
{quiz}

Check from an expert English Writer of the above quiz:"""

In [ ]:
quiz_evaluation_prompt=PromptTemplate(input_variables=["subject", "quiz"], template=TEMPLATE2)

In [ ]:
review_chain = quiz_evaluation_prompt | llm | StrOutputParser()

In [ ]:
generate_evaluate_chain = (
    RunnablePassthrough.assign(quiz=quiz_chain)
    | RunnablePassthrough.assign(review=review_chain)
)

In [ ]:
from pathlib import Path
file_path=r"D:\Python\GenAI\McqGen\data.txt"

In [ ]:
with open(file_path, 'r') as file:
    TEXT = file.read()

In [ ]:
json.dumps(RESPONSE_JSON)

In [ ]:
NUMBER=5 
SUBJECT="data science"
TONE="simple"

In [ ]:
with get_openai_callback() as cb:
    response = generate_evaluate_chain.invoke({
        "text": TEXT,
        "number": NUMBER,
        "subject": SUBJECT,
        "tone": TONE,
        "response_json": json.dumps(RESPONSE_JSON)
    })

print(response)
print("Total Tokens:", cb.total_tokens)
print("Prompt Tokens:", cb.prompt_tokens)
print("Completion Tokens:", cb.completion_tokens)
print("Total Cost:", cb.total_cost)

In [ ]:
response

In [ ]:
quiz = response.get("quiz")

quiz = quiz.replace("```json", "").replace("```", "").strip()



In [ ]:
quiz

In [ ]:
quiz_data = json.loads(quiz)

print(json.dumps(quiz_data, indent=4))

In [ ]:
quiz_table_data = []
for key, value in quiz_data.items():
    mcq = value["mcq"]
    options = " | ".join(
        [
            f"{option}: {option_value}"
            for option, option_value in value["options"].items()
            ]
        )
    correct = value["correct"]
    quiz_table_data.append({"MCQ": mcq, "Choices": options, "Correct": correct})

In [ ]:
quiz_table_data

In [ ]:
quiz = pd.DataFrame(quiz_table_data)

In [ ]:
import os

print(os.path.exists("data_science_mcq_quiz.csv"))

In [ ]:

quiz.to_csv("data_science_mcq_quiz.csv", index=False)